# Run a greedy episode from a saved fitted-Q policy

Loads `artifacts/fitted_q_policy.joblib` and runs one **deployment** episode
(epsilon = 0, no replay collection, no retraining).

**Prerequisite:** run `adult_fitted_q_learning_run.ipynb` first so a current
bundle (version 3) exists. If you changed the Python module, restart the kernel
before running.

**No retraining needed** for rollout — the artifact already contains the fitted
Q-regressors. You rebuild the Adult data split from the saved `config`, then
run the greedy policy.

**Budget caveat:** the Q-models were trained with specific budget caps baked into
state features (`acq_budget_frac`, etc.). You can change budgets at inference time
without retraining, but if the new caps differ a lot from training the policy may
behave suboptimally because the state normalization no longer matches what it saw.

In [7]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from adult_prototype_fitted_q_learning import (
    DEFAULT_Q_POLICY_ARTIFACT_PATH,
    build_experiment,
    build_final_episode_tables,
    load_q_policy,
    run_greedy_episode_with_budgets,
    score_on_test,
)

ARTIFACT_PATH = PROJECT_ROOT / DEFAULT_Q_POLICY_ARTIFACT_PATH

In [8]:
bundle = load_q_policy(ARTIFACT_PATH)
q_models = bundle["q_models"]
q_model_is_fitted = bundle["q_model_is_fitted"]
training_config = bundle["config"]

print(f"Loaded artifact: {ARTIFACT_PATH}")
print(
    "Training budgets: "
    f"acquire={training_config['acquisition_budget']}, "
    f"retrain={training_config['retrain_budget']}, "
    f"evaluate={training_config['evaluation_budget']}"
)
print(
    "Fitted Q-models: "
    f"{[action for action, fitted in q_model_is_fitted.items() if fitted]}"
)

ValueError: Unsupported bundle version 12; expected 11.

## Custom budgets

Edit the three values below, then run the next cells. These override the rollout
`config` budgets only for this episode.

In [3]:
CUSTOM_ACQUISITION_BUDGET = 6
CUSTOM_RETRAIN_BUDGET = 3
CUSTOM_EVALUATION_BUDGET = 5

EPISODE_SEED = (
    training_config["random_seed"] + training_config["final_episode_seed_offset"]
)

In [4]:
experiment = build_experiment(training_config)

Adult rows used: 5,000; features: 14; positive fraction: 0.239


In [5]:
episode_result = run_greedy_episode_with_budgets(
    q_models=q_models,
    q_model_is_fitted=q_model_is_fitted,
    experiment=experiment,
    acquisition_budget=CUSTOM_ACQUISITION_BUDGET,
    retrain_budget=CUSTOM_RETRAIN_BUDGET,
    evaluation_budget=CUSTOM_EVALUATION_BUDGET,
    episode_seed=EPISODE_SEED,
)
test_scores = score_on_test(episode_result["model"], experiment)
action_results, episode_showcase, reward_comparison = build_final_episode_tables(
    episode_result,
    test_scores,
)

print(
    "Action sequence: "
    + " → ".join(action_results["action"])
)
print(
    f"\nBudgets used: acquire={CUSTOM_ACQUISITION_BUDGET}, "
    f"retrain={CUSTOM_RETRAIN_BUDGET}, evaluate={CUSTOM_EVALUATION_BUDGET}"
)
print(f"Hidden validation AUC: {episode_result['terminal_auc']:.4f}")
print(f"Test AUC: {test_scores['test_auc']:.4f}")
print(f"Test accuracy: {test_scores['test_accuracy']:.4f}")

TypeError: build_final_episode_tables() missing 1 required positional argument: 'config'

In [ ]:
episode_showcase

In [ ]:
reward_comparison

In [6]:
cost_long = action_results.melt(
    id_vars=["action_number", "action"],
    value_vars=[
        "acquisition_cost_so_far",
        "retrain_cost_so_far",
        "evaluation_cost_so_far",
    ],
    var_name="cost_type",
    value_name="cost_so_far",
)
cost_long["cost_type"] = cost_long["cost_type"].str.replace("_cost_so_far", "")

px.line(
    cost_long,
    x="action_number",
    y="cost_so_far",
    color="cost_type",
    markers=True,
    title="Budget spend over the episode",
    labels={
        "action_number": "Step",
        "cost_so_far": "Cumulative cost",
        "cost_type": "Budget",
    },
)

NameError: name 'action_results' is not defined

In [ ]:
px.line(
    action_results,
    x="action_number",
    y=["q_value_at_action", "cumulative_actual_reward"],
    markers=True,
    title="Perceived Q vs cumulative actual reward",
    labels={
        "action_number": "Step",
        "value": "Value",
        "variable": "Series",
    },
)